# Strands Agent Deployment with Observability — FSI Edition

This lab demonstrates deploying a Strands Agent to Bedrock AgentCore Runtime with full observability — critical for FSI compliance and audit trails.

## Overview

In this lab, you will:
- Deploy a Strands Agent with FSI tools to AgentCore Runtime
- Invoke the deployed agent via boto3
- View traces and spans in CloudWatch (GenAI Observability)
- Understand how observability supports FSI compliance

## Why Observability for FSI?

Regulators require financial institutions to:
- **Audit every decision** — What did the agent do and why?
- **Trace data flow** — Where did the data come from?
- **Monitor performance** — Is the agent responding within SLAs?
- **Detect anomalies** — Is the agent behaving unexpectedly?

## Prerequisites

⚠️ **Important**: Enable [CloudWatch Transaction Search](https://console.aws.amazon.com/cloudwatch/home#logsV2:transaction-search) and set X-Ray trace indexing to **100%** before starting this lab.

In [ ]:
import os
#os.environ['AWS_ACCESS_KEY_ID'] = ''
#os.environ['AWS_SECRET_ACCESS_KEY'] = ''
#os.environ['AWS_SESSION_TOKEN'] = ''
#os.environ['AWS_REGION'] = ''

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

In [1]:
import boto3
region = boto3.session.Session().region_name
NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'): NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'): NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'
print(f'Region: {region}, Model: {NOVA_PRO_MODEL_ID}')

Region: ap-southeast-2, Model: apac.amazon.nova-pro-v1:0


## Step 1: Create the Agent Application

We'll create a Python file that defines our FSI agent with tools, wrapped in the `BedrockAgentCoreApp` class for deployment.

In [4]:
%%writefile strands_agent.py
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import os

region = os.environ.get('AWS_REGION', 'ap-southeast-2')
MODEL_ID = 'apac.amazon.nova-pro-v1:0' if region.startswith('ap') else 'us.amazon.nova-pro-v1:0'

@tool
def validate_transaction(amount: float, merchant_category: str = 'general') -> str:
    '''Validate a transaction against risk rules.
    Args:
        amount: Transaction amount in AUD
        merchant_category: Category (general, crypto, gambling, high_risk)
    '''
    risk_score = 0
    flags = []
    if amount > 10000: flags.append('HIGH_VALUE'); risk_score += 3
    if merchant_category in ('crypto', 'gambling'): flags.append(f'HIGH_RISK_{merchant_category.upper()}'); risk_score += 4
    decision = 'BLOCKED' if risk_score >= 7 else 'REVIEW' if risk_score >= 4 else 'APPROVED'
    return f'Decision: {decision} | Risk: {risk_score}/10 | Flags: {flags}'

@tool
def get_account_balance(account_id: str) -> str:
    '''Get account balance.
    Args:
        account_id: Account identifier
    '''
    balances = {'ACC-001': '$125,430.50', 'ACC-002': '$2,340,000.00', 'ACC-003': '$45,200.75'}
    return f'Account {account_id}: Balance {balances.get(account_id, "Not found")}'

agent = Agent(
    model=BedrockModel(model_id=MODEL_ID, max_tokens=4096),
    system_prompt='You are a banking operations assistant. Validate transactions and check accounts. Be concise.',
    tools=[validate_transaction, get_account_balance],
)

app = BedrockAgentCoreApp(agent=agent)

if __name__ == '__main__':
    app.serve()

Overwriting strands_agent.py


In [5]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
bedrock-agentcore

Overwriting requirements.txt


## Step 2: Deploy to AgentCore Runtime

In [6]:
from bedrock_agentcore_starter_toolkit import Runtime
import boto3

region = boto3.session.Session().region_name

agentcore_runtime = Runtime()

print("Configuring and deploying FSI agent...")
response = agentcore_runtime.configure(
    entrypoint="strands_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="fsi_observability_agent",
    non_interactive=True,
)
print("Configuration completed")

print("Launching deployment (3-5 minutes)...")
launch_result = agentcore_runtime.launch()
runtime_id = launch_result.agent_id
runtime_arn = launch_result.agent_arn
print(f"Deployed! Runtime ID: {runtime_id}")
print(f"ARN: {runtime_arn}")


ImportError: cannot import name 'deploy' from 'bedrock_agentcore.runtime' (/Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/.venv/lib/python3.11/site-packages/bedrock_agentcore/runtime/__init__.py)

## Step 3: Invoke the Deployed Agent

In [ ]:
from bedrock_agentcore._utils import endpoints
import json

dp_endpoint = endpoints.get_data_plane_endpoint(region)
dp_client = boto3.client("bedrock-agentcore", region_name=region, endpoint_url=dp_endpoint)

# Invoke the deployed agent
encoded_arn = runtime_arn.replace(":", "%3A").replace("/", "%2F")
response = dp_client.invoke_runtime(
    runtimeIdentifier=runtime_id,
    payload=json.dumps({"prompt": "Validate a $20,000 transaction to a crypto exchange and check balance for ACC-001"}),
)

result = json.loads(response["body"].read())
print(result.get("response", result))


## Step 4: View Traces in CloudWatch

Navigate to the [CloudWatch Console → Application Signals → Traces](https://console.aws.amazon.com/cloudwatch/home#xray:traces) to see:

- **End-to-end trace** of the agent invocation
- **Spans** for each tool call (validate_transaction, get_account_balance)
- **Latency** breakdown per component
- **Model invocation** details (tokens, duration)

You can also check the **AgentCore tab** in CloudWatch for a fleet-level view.

### What the Audit Trail Shows (FSI Compliance)

| Trace Element | Compliance Value |
|--------------|------------------|
| Request timestamp | When was the decision made? |
| Tool calls | What data was consulted? |
| Model reasoning | Why was this decision reached? |
| Response | What was communicated? |
| Latency | Was it within SLA? |

## Cleanup (Optional)

In [ ]:
# Uncomment to clean up
# agentcore_runtime.delete()
# print("✅ Runtime deleted")


## Summary

- ✅ Deployed FSI agent to AgentCore Runtime
- ✅ Invoked via boto3 with IAM authentication
- ✅ Viewed traces in CloudWatch (full audit trail)
- ✅ Understood observability for FSI compliance

### Next: Lab 06 — Memory
We'll add persistent memory so the agent remembers client context across sessions.